In [1]:
import ollama
import numpy as np

In [2]:
embedded_model="http://hf.co/CompendiumLabs/bge-base-en-v1.5-gguf"
language_model="http://hf.co/bartowski/Llama-3.2-1B-Instruct-GGUF"


In [3]:
vector_DB=[]
def add_chunk_to_database(chunk):
    try:
        response=ollama.embed(model=embedded_model,input=chunk)
        if "embedding" in response:
            embedding=response["embedding"]
        elif "embeddings" in response:
            embeddings=response["embeddings"]
            if isinstance(embeddings[0],list):
                embedding=embeddings[0]
            else:
                embedding=embeddings
        else:
            print("chunk and vector adding failed")
            return 
        embedding_np=np.array(embedding,dtype="float")
        vector_DB.append((chunk,embedding_np))
        print(f"chunk and vector successfully added with shape{embedding_np.shape}")
    except Exception as e:
        print(f"{e}")

In [4]:
def cosine_similarity(a,b):
    a_np=np.array(a,dtype="float")
    b_np=np.array(b,dtype="float")
    dot_product=np.dot(a_np,b_np)
    norm_a=np.linalg.norm(a_np)
    norm_b=np.linalg.norm(b_np)
    if norm_a==0 or norm_b==0:
        return 0
    else:
        return dot_product/(norm_a*norm_b)

In [5]:

def retrive(query,top_n=3):
    try:
        responce=ollama.embed(model=embedded_model,input=query)
        if "embedding" in responce:
            embedding_query=responce["embedding"]
        elif "embeddings" in responce:
            embedding_query=responce["embeddings"][0]
        else:
            print("vector not returned")
            return
        embedded_query_np=np.array(embedding_query,dtype="float")
        similarity=[]
        for chunk,embedding in vector_DB:
            sim=cosine_similarity(embedded_query_np,embedding)
            similarity.append((chunk,sim))
        similarity.sort(key=lambda x:x[1],reverse=True)
        return similarity[:top_n]
    except Exception as e:
        print(f"error is {e}")

In [6]:
def main():
    print("loading dataset")
    with open("data.txt","r") as file1:
        dataset=file1.readlines()
    dataset=[i.strip() for i in dataset]
    print(f"loaded {len(dataset)} entries")
    print("Indexing dataset(this might take few minute)....")
    for i,chunk in enumerate(dataset):
        add_chunk_to_database(chunk)
        print(f'Added chunk {i+1}/ {len(dataset)} to the database')
    while True:
        query_input=input("Ask me about the grren bottle(or type EXiT to stop)")
        if query_input.lower()=="exit":
            break
        print("\nretriving Knowledge")
        retiving_knowledge=retrive(query_input)
        print("\nretivied knowledge")
        for chunk,similarity in retiving_knowledge:
            print(f"-{chunk}")
        instraction_prompt= f''' You are a helpful chatbot that knows about institue Luminar Technolab. Use only the following pieces of context to answer the question. 
        Dont make up any new information:{"\n".join([f'-{chunk}'for chunk,similarity in retiving_knowledge])}'''
        print("\nchatbot responce")
        s=ollama.chat(model=language_model,messages=
                      [{"role":"system","content":instraction_prompt},
                       {"role":"user","content":query_input}],stream=True)
        print("\nchatbot responce")
        for chunk in s:
            print(chunk["message"]["content"],end="",flush=True)
if __name__=="__main__":
    try:
        main()
    except Exception as e:
        print(f"\n error in {e}")

loading dataset
loaded 10 entries
Indexing dataset(this might take few minute)....
chunk and vector successfully added with shape(768,)
Added chunk 1/ 10 to the database
chunk and vector successfully added with shape(768,)
Added chunk 2/ 10 to the database
chunk and vector successfully added with shape(768,)
Added chunk 3/ 10 to the database
chunk and vector successfully added with shape(768,)
Added chunk 4/ 10 to the database
chunk and vector successfully added with shape(768,)
Added chunk 5/ 10 to the database
chunk and vector successfully added with shape(768,)
Added chunk 6/ 10 to the database
chunk and vector successfully added with shape(768,)
Added chunk 7/ 10 to the database
chunk and vector successfully added with shape(768,)
Added chunk 8/ 10 to the database
chunk and vector successfully added with shape(768,)
Added chunk 9/ 10 to the database
chunk and vector successfully added with shape(768,)
Added chunk 10/ 10 to the database

retriving Knowledge

retivied knowledge
-The 

In [7]:
print(vector_DB)

[('The green bottle is made of recycled glass, giving it a slightly textured surface and a natural emerald hue.",', array([-1.70498300e-02, -5.48362660e-02, -3.71238480e-02,  3.53597330e-02,
        5.25808370e-02, -1.30606540e-02, -3.26160750e-02,  5.13027700e-02,
       -2.50829830e-02, -5.95564100e-02, -6.56053050e-02, -1.42902900e-02,
       -6.23094550e-02,  2.92828330e-02,  2.70949360e-02,  3.72145960e-02,
        5.47060040e-02,  1.94317080e-02,  4.71956000e-02,  3.28023570e-03,
       -1.68367030e-02,  3.47742020e-03,  2.93603650e-02, -1.58623330e-02,
        3.46666300e-02, -1.24936100e-02,  3.91565150e-02,  8.22674860e-02,
       -3.58083200e-02,  6.10112620e-02,  6.11949700e-02,  2.43913120e-02,
        2.59813000e-03, -2.02500930e-02,  5.75565660e-02, -1.28022510e-03,
       -5.49453650e-02, -1.21009685e-02, -1.19969964e-01, -2.98521650e-02,
        1.99056030e-02,  5.44077600e-02,  9.70928000e-03,  8.15808300e-03,
       -8.08633100e-03,  8.83475200e-02,  1.77085960e-03,  